# ¿Cuánto se puede pronosticar usando el historial del índice?
Pronóstico retrospectivo directo a cuatro horizontes. Entrenamiento inicial antes de 1995, validación 1995–2009 y orígenes de prueba desde 2010. Las etiquetas de entrenamiento y validación terminan antes del siguiente tramo.
Ejecuta las celdas en orden. Las tablas CSV y figuras se guardan en la carpeta de trabajo.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/gretadive/correlacion_actividad_solar_elnino/751cd7c187616d8b7ed7b175280dc80a4e999a29/data/processed/dataset_integrado_precipitacion.csv"
archivo = Path("datos.csv")
df = pd.read_csv(archivo if archivo.exists() else url, parse_dates=["fecha"])
df = df.set_index("fecha").sort_index().asfreq("MS")
variables = ["ICEN", "ONI", "precipitacion_mm"]
assert len(df) == 917 and df[variables + ["SSN"]].notna().all().all()
desfases = [0, 3, 6, 12, 24]


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
# El origen t usa índices hasta t-1 por la ventana centrada trimestral.
# Supuesto retrospectivo: el índice de t-1 está disponible al cierre de t.
def experimento(incluir_solar):
    filas = []
    pronosticos = []
    for v in ["ICEN","ONI"]:
        for h in [3,6,12,24]:
            entradas = {f"{v}_{i}":df[v].shift(i) for i in range(1,13)}
            if incluir_solar:
                entradas.update({f"SSN_{i}":df.SSN.shift(i) for i in range(1,13)})
            X = pd.DataFrame(entradas)
            y = df[v].shift(-h)
            destino = df.index + pd.DateOffset(months=h)
            valido = X.notna().all(axis=1) & y.notna()
            train = valido & (destino < "1995-01-01")
            val = valido & (df.index >= "1995-01-01") & (destino < "2010-01-01")
            test = valido & (df.index >= "2010-01-01")
            errores = []
            for alfa in [0.1,1,10,100]:
                modelo = make_pipeline(StandardScaler(),Ridge(alpha=alfa))
                modelo.fit(X[train],y[train])
                errores.append(np.mean((modelo.predict(X[val])-y[val])**2))
            alfa = [0.1,1,10,100][np.argmin(errores)]
            modelo = make_pipeline(StandardScaler(),Ridge(alpha=alfa))
            ajuste = valido & (destino < "2010-01-01")
            modelo.fit(X[ajuste],y[ajuste])
            modelos = {"Ridge con SSN" if incluir_solar else "Ridge sin SSN":modelo.predict(X[test])}
            if not incluir_solar:
                modelos["Persistencia"] = X.loc[test,f"{v}_1"].to_numpy()
            for etiqueta, pred in modelos.items():
                error = pred-y[test].to_numpy()
                filas.append([v,h,etiqueta,len(pred),np.abs(error).mean(),np.sqrt((error**2).mean()),alfa if etiqueta!="Persistencia" else None])
                pronosticos.extend(zip([v]*len(pred),[h]*len(pred),[etiqueta]*len(pred),df.index[test],destino[test],y[test],pred))
    return (pd.DataFrame(filas,columns=["indice","h","modelo","n","MAE","RMSE","alfa"]),
            pd.DataFrame(pronosticos,columns=["indice","h","modelo","origen","destino","observado","estimado"]))

metricas, series = experimento(False)
metricas.to_csv("14_referencia.csv",index=False)
series.to_csv("14_pronosticos.csv",index=False)
print(metricas.round(3).to_string(index=False))

indice  h        modelo   n   MAE  RMSE  alfa
  ICEN  3 Ridge sin SSN 194 0.548 0.703   0.1
  ICEN  3  Persistencia 194 0.724 0.904   NaN
  ICEN  6 Ridge sin SSN 191 0.768 0.951   0.1
  ICEN  6  Persistencia 191 1.012 1.262   NaN
  ICEN 12 Ridge sin SSN 185 0.832 1.013   0.1
  ICEN 12  Persistencia 185 1.129 1.481   NaN
  ICEN 24 Ridge sin SSN 173 0.864 1.039 100.0
  ICEN 24  Persistencia 173 1.235 1.597   NaN
   ONI  3 Ridge sin SSN 194 0.360 0.457   0.1
   ONI  3  Persistencia 194 0.544 0.685   NaN
   ONI  6 Ridge sin SSN 191 0.537 0.688   0.1
   ONI  6  Persistencia 191 0.763 0.992   NaN
   ONI 12 Ridge sin SSN 185 0.596 0.773   0.1
   ONI 12  Persistencia 185 0.893 1.159   NaN
   ONI 24 Ridge sin SSN 173 0.572 0.757  10.0
   ONI 24  Persistencia 173 1.133 1.373   NaN
